# EXO-INSIGHT — Milestone 3: Data Preprocessing & Cleaning

## Objective
Prepare a reliable, research-ready working dataset from NASA's `PSCompPars.csv` while preserving the original raw file.

**Input:** `DATA/raw/PSCompPars.csv`  
**Outputs:** preprocessing reports and a separate working dataset in `DATA/processed/`.

**Principles**
- Never overwrite the raw NASA dataset.
- Missing measurements are not zero and will not be imputed without justification.
- Do not apply arbitrary physical thresholds or candidate-selection cut-offs.
- Document every transformation and validation decision.

## 1. Project setup and dataset loading
Load the original NASA dataset and define the output folder.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

PROJECT_DIR = Path(r"S:\Aivancity\PGE 3\Coding for AI\EXO INSIGHTS")
RAW_FILE = PROJECT_DIR / "DATA" / "raw" / "PSCompPars.csv"
PROCESSED_DIR = PROJECT_DIR / "DATA" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_FILE.exists():
    raise FileNotFoundError(f"NASA dataset not found: {RAW_FILE}")

df_raw = pd.read_csv(RAW_FILE, comment="#", low_memory=False)

print("Dataset loaded successfully")
print("Source:", RAW_FILE)
print("Rows:", df_raw.shape[0])
print("Columns:", df_raw.shape[1])
print("Size (MB):", round(RAW_FILE.stat().st_size / (1024 * 1024), 2))
display(df_raw.head())

Dataset loaded successfully
Source: S:\Aivancity\PGE 3\Coding for AI\EXO INSIGHTS\DATA\raw\PSCompPars.csv
Rows: 6451
Columns: 71
Size (MB): 3.4


,pl_name,hostname,sy_snum,sy_pnum,discoverymethod,disc_year,disc_facility,pl_orbper,pl_orbpererr1,pl_orbpererr2,...,sy_disterr2,sy_vmag,sy_vmagerr1,sy_vmagerr2,sy_kmag,sy_kmagerr1,sy_kmagerr2,sy_gaiamag,sy_gaiamagerr1,sy_gaiamagerr2
0,11 Com b,11 Com,2,1,Radial Velocity,2007,Xinglong Station,323.21000,0.06,-0.05,...,-1.9238,4.72307,0.023,-0.023,2.282,0.346,-0.346,4.44038,0.003848,-0.003848
1,11 UMi b,11 UMi,1,1,Radial Velocity,2009,Thueringer Landessternwarte Tautenburg,516.21997,3.20,-3.20,...,-1.9765,5.01300,0.005,-0.005,1.939,0.270,-0.270,4.56216,0.003903,-0.003903
2,14 And b,14 And,1,1,Radial Velocity,2008,Okayama Astrophysical Observatory,186.76000,0.11,-0.12,...,-0.7140,5.23133,0.023,-0.023,2.331,0.240,-0.240,4.91781,0.002826,-0.002826
3,14 Her b,14 Her,1,2,Radial Velocity,2002,W. M. Keck Observatory,1766.41000,0.67,-0.68,...,-0.0073,6.61935,0.023,-0.023,4.714,0.016,-0.016,6.38300,0.000351,-0.000351
4,16 Cyg B b,16 Cyg B,3,1,Radial Velocity,1996,Multiple Observatories,798.50000,1.00,-1.00,...,-0.0111,6.21500,0.016,-0.016,4.651,0.016,-0.016,6.06428,0.000603,-0.000603


## 2. Select research-relevant columns
Create a separate working copy with planet identifiers, discovery information, planetary measurements, host-star measurements, distance, and mass provenance. `st_lum` was unavailable in the Milestone 2 profiling results.

In [2]:
selected_columns = [
    "pl_name", "hostname", "discoverymethod", "disc_year",
    "pl_rade", "pl_bmasse", "pl_eqt", "pl_insol",
    "pl_orbper", "pl_orbsmax", "st_teff", "st_rad",
    "st_mass", "sy_dist", "pl_bmassprov"
]

available_columns = [c for c in selected_columns if c in df_raw.columns]
unavailable_columns = [c for c in selected_columns if c not in df_raw.columns]
df = df_raw.loc[:, available_columns].copy()

print("Selected columns:", len(available_columns))
print("Unavailable requested columns:", unavailable_columns)
print("Working dataset shape:", df.shape)
display(df.head())

Selected columns: 15
Unavailable requested columns: []
Working dataset shape: (6451, 15)


,pl_name,hostname,discoverymethod,disc_year,pl_rade,pl_bmasse,pl_eqt,pl_insol,pl_orbper,pl_orbsmax,st_teff,st_rad,st_mass,sy_dist,pl_bmassprov
0,11 Com b,11 Com,Radial Velocity,2007,12.2,4914.898486,803.26,68.5387,323.21000,1.178,4874.0,13.76,2.09,93.1846,Msini
1,11 UMi b,11 UMi,Radial Velocity,2009,12.3,4684.814200,896.43,114.8502,516.21997,1.530,4213.0,29.79,2.78,125.3210,Msini
2,14 And b,14 And,Radial Velocity,2008,13.1,1131.151301,909.92,115.1634,186.76000,0.775,4888.0,11.55,1.78,75.4392,Msini
3,14 Her b,14 Her,Radial Velocity,2002,12.5,2828.672822,147.32,0.0873,1766.41000,2.839,5338.0,0.93,0.97,17.9323,Mass
4,16 Cyg B b,16 Cyg B,Radial Velocity,1996,13.5,565.737400,228.76,0.4540,798.50000,1.660,5750.0,1.13,1.08,21.1397,Msini


## 3. Missing-value analysis
Quantify missingness. Do not delete rows or fill values at this stage.

In [3]:
missing_report = pd.DataFrame({
    "column": df.columns,
    "missing_count": df.isna().sum().values,
    "missing_percentage": (df.isna().mean().values * 100).round(2),
    "available_count": df.notna().sum().values
}).sort_values("missing_percentage", ascending=False)

display(missing_report)
missing_report.to_csv(PROCESSED_DIR / "milestone3_missingness_report.csv", index=False)

,column,missing_count,missing_percentage,available_count
7,pl_insol,596,9.24,5855
6,pl_eqt,540,8.37,5911
9,pl_orbsmax,495,7.67,5956
8,pl_orbper,354,5.49,6097
11,st_rad,332,5.15,6119
10,st_teff,307,4.76,6144
4,pl_rade,50,0.78,6401
5,pl_bmasse,30,0.47,6421
13,sy_dist,28,0.43,6423
12,st_mass,9,0.14,6442


## 4. Validate and convert data types
Convert expected measurement and year fields to numeric. Unparseable values become missing and are counted.

In [4]:
numeric_columns = [
    "disc_year", "pl_rade", "pl_bmasse", "pl_eqt", "pl_insol",
    "pl_orbper", "pl_orbsmax", "st_teff", "st_rad", "st_mass", "sy_dist"
]
numeric_columns = [c for c in numeric_columns if c in df.columns]

conversion_records = []
for col in numeric_columns:
    before_non_null = int(df[col].notna().sum())
    converted = pd.to_numeric(df[col], errors="coerce")
    after_non_null = int(converted.notna().sum())
    conversion_records.append({
        "column": col,
        "non_null_before": before_non_null,
        "non_null_after": after_non_null,
        "new_missing_from_conversion": before_non_null - after_non_null
    })
    df[col] = converted

type_report = pd.DataFrame(conversion_records)
display(type_report)
display(df.dtypes.rename("dtype").to_frame())

,column,non_null_before,non_null_after,new_missing_from_conversion
0,disc_year,6451,6451,0
1,pl_rade,6401,6401,0
2,pl_bmasse,6421,6421,0
3,pl_eqt,5911,5911,0
4,pl_insol,5855,5855,0
5,pl_orbper,6097,6097,0
6,pl_orbsmax,5956,5956,0
7,st_teff,6144,6144,0
8,st_rad,6119,6119,0
9,st_mass,6442,6442,0


,dtype
pl_name,object
hostname,object
discoverymethod,object
disc_year,int64
pl_rade,float64
pl_bmasse,float64
pl_eqt,float64
pl_insol,float64
pl_orbper,float64
pl_orbsmax,float64


## 5. Duplicate and consistency checks
Check repeated planet names and exact duplicate rows. These are diagnostic checks; no rows are automatically removed.

In [5]:
exact_duplicate_count = int(df.duplicated().sum())

if "pl_name" in df.columns:
    repeated_name_mask = df["pl_name"].notna() & df["pl_name"].duplicated(keep=False)
    repeated_name_rows = int(repeated_name_mask.sum())
    repeated_names = df.loc[repeated_name_mask, "pl_name"].drop_duplicates().sort_values().tolist()
else:
    repeated_name_rows = 0
    repeated_names = []

duplicate_report = pd.DataFrame([
    {"check": "Exact duplicate rows", "count": exact_duplicate_count},
    {"check": "Rows with repeated planet names", "count": repeated_name_rows}
])
display(duplicate_report)
print("Repeated planet names (first 30):", repeated_names[:30])

,check,count
0,Exact duplicate rows,0
1,Rows with repeated planet names,0


Repeated planet names (first 30): []


## 6. Physical-value screening
Flag zero, negative, and missing values for selected positive-valued measurements. This is screening, not automatic deletion; review the NASA field definitions before deciding how to treat flagged values.

In [6]:
positive_measurement_columns = [
    "pl_rade", "pl_bmasse", "pl_eqt", "pl_insol",
    "pl_orbper", "pl_orbsmax", "st_teff", "st_rad", "st_mass", "sy_dist"
]
positive_measurement_columns = [c for c in positive_measurement_columns if c in df.columns]

range_records = []
for col in positive_measurement_columns:
    values = df[col]
    range_records.append({
        "column": col,
        "missing": int(values.isna().sum()),
        "zero_count": int((values == 0).sum()),
        "negative_count": int((values < 0).sum()),
        "positive_count": int((values > 0).sum())
    })

physical_screen_report = pd.DataFrame(range_records)
display(physical_screen_report)
physical_screen_report.to_csv(
    PROCESSED_DIR / "milestone3_physical_screening_report.csv", index=False
)

,column,missing,zero_count,negative_count,positive_count
0,pl_rade,50,0,0,6401
1,pl_bmasse,30,0,0,6421
2,pl_eqt,540,0,0,5911
3,pl_insol,596,46,0,5809
4,pl_orbper,354,0,0,6097
5,pl_orbsmax,495,0,0,5956
6,st_teff,307,0,0,6144
7,st_rad,332,0,0,6119
8,st_mass,9,0,0,6442
9,sy_dist,28,0,0,6423


## 7. Missing-value treatment strategy
For this preprocessing pass, retain missing values. Do not replace them with zero, mean, median, or estimates. Later analyses can create task-specific subsets.

In [7]:
treatment_register = pd.DataFrame({
    "column": df.columns,
    "missing_count": [int(df[c].isna().sum()) for c in df.columns],
    "treatment": ["Retain as missing; no imputation in this stage" for _ in df.columns],
    "reason": ["Avoid introducing unsupported values into scientific measurements" for _ in df.columns]
})
display(treatment_register)
treatment_register.to_csv(
    PROCESSED_DIR / "milestone3_missing_value_treatment.csv", index=False
)

,column,missing_count,treatment,reason
0,pl_name,0,Retain as missing; no imputation in this stage,Avoid introducing unsupported values into scie...
1,hostname,0,Retain as missing; no imputation in this stage,Avoid introducing unsupported values into scie...
2,discoverymethod,0,Retain as missing; no imputation in this stage,Avoid introducing unsupported values into scie...
3,disc_year,0,Retain as missing; no imputation in this stage,Avoid introducing unsupported values into scie...
4,pl_rade,50,Retain as missing; no imputation in this stage,Avoid introducing unsupported values into scie...
5,pl_bmasse,30,Retain as missing; no imputation in this stage,Avoid introducing unsupported values into scie...
6,pl_eqt,540,Retain as missing; no imputation in this stage,Avoid introducing unsupported values into scie...
7,pl_insol,596,Retain as missing; no imputation in this stage,Avoid introducing unsupported values into scie...
8,pl_orbper,354,Retain as missing; no imputation in this stage,Avoid introducing unsupported values into scie...
9,pl_orbsmax,495,Retain as missing; no imputation in this stage,Avoid introducing unsupported values into scie...


## 8. Research filtering and candidate selection — pending approved criteria
Do not apply candidate-selection thresholds until the research protocol specifies them. A potentially temperate candidate cannot be identified responsibly using arbitrary cut-offs chosen during data cleaning. The current preprocessed dataset remains inclusive.

In [8]:
candidate_screening_fields = [
    "pl_eqt", "pl_insol", "pl_orbsmax", "pl_rade",
    "pl_bmasse", "st_teff", "st_rad", "st_mass"
]
candidate_screening_fields = [c for c in candidate_screening_fields if c in df.columns]

print("Potential screening fields available:")
for col in candidate_screening_fields:
    print("-", col)

print("\nNo candidate thresholds have been applied.")
print("All records remain in the working dataset.")

Potential screening fields available:
- pl_eqt
- pl_insol
- pl_orbsmax
- pl_rade
- pl_bmasse
- st_teff
- st_rad
- st_mass

No candidate thresholds have been applied.
All records remain in the working dataset.


## 9. Cleaning log and before/after validation
Summarize the operations performed and compare original and working row counts.

In [9]:
cleaning_log = pd.DataFrame([
    {"step": "Load source", "action": "Loaded NASA source without modifying it",
     "rows_before": len(df_raw), "rows_after": len(df_raw), "rows_affected": 0},
    {"step": "Column selection", "action": "Created working copy with available research-relevant columns",
     "rows_before": len(df_raw), "rows_after": len(df), "rows_affected": 0},
    {"step": "Numeric conversion", "action": "Converted selected fields; unparseable text becomes missing",
     "rows_before": len(df), "rows_after": len(df), "rows_affected": 0},
    {"step": "Missing values", "action": "Retained missing values; no imputation or row deletion",
     "rows_before": len(df), "rows_after": len(df), "rows_affected": 0},
    {"step": "Physical screening", "action": "Flagged zero/negative values; no automatic deletion",
     "rows_before": len(df), "rows_after": len(df), "rows_affected": 0}
])
display(cleaning_log)
cleaning_log.to_csv(PROCESSED_DIR / "milestone3_cleaning_log.csv", index=False)

validation_report = pd.DataFrame([
    {"metric": "Original row count", "value": len(df_raw)},
    {"metric": "Working row count", "value": len(df)},
    {"metric": "Selected column count", "value": len(df.columns)},
    {"metric": "Exact duplicate rows", "value": int(df.duplicated().sum())},
    {"metric": "Rows removed in this notebook", "value": len(df_raw) - len(df)}
])
display(validation_report)
validation_report.to_csv(PROCESSED_DIR / "milestone3_validation_report.csv", index=False)

,step,action,rows_before,rows_after,rows_affected
0,Load source,Loaded NASA source without modifying it,6451,6451,0
1,Column selection,Created working copy with available research-r...,6451,6451,0
2,Numeric conversion,Converted selected fields; unparseable text be...,6451,6451,0
3,Missing values,Retained missing values; no imputation or row ...,6451,6451,0
4,Physical screening,Flagged zero/negative values; no automatic del...,6451,6451,0


,metric,value
0,Original row count,6451
1,Working row count,6451
2,Selected column count,15
3,Exact duplicate rows,0
4,Rows removed in this notebook,0


## 10. Export the preprocessed dataset and reports
Export the selected, type-checked working dataset. This is not a final candidate list; it preserves all rows and missing values.

In [10]:
CLEANED_FILE = PROCESSED_DIR / "exo_insight_preprocessed.csv"
df.to_csv(CLEANED_FILE, index=False)

print("MILESTONE 3 EXPORTS")
print("=" * 55)
for filename in [
    "exo_insight_preprocessed.csv",
    "milestone3_missingness_report.csv",
    "milestone3_physical_screening_report.csv",
    "milestone3_missing_value_treatment.csv",
    "milestone3_cleaning_log.csv",
    "milestone3_validation_report.csv"
]:
    print(PROCESSED_DIR / filename)

print("\nFinal preprocessed shape:", df.shape)
print("Raw source file was not overwritten.")

MILESTONE 3 EXPORTS
S:\Aivancity\PGE 3\Coding for AI\EXO INSIGHTS\DATA\processed\exo_insight_preprocessed.csv
S:\Aivancity\PGE 3\Coding for AI\EXO INSIGHTS\DATA\processed\milestone3_missingness_report.csv
S:\Aivancity\PGE 3\Coding for AI\EXO INSIGHTS\DATA\processed\milestone3_physical_screening_report.csv
S:\Aivancity\PGE 3\Coding for AI\EXO INSIGHTS\DATA\processed\milestone3_missing_value_treatment.csv
S:\Aivancity\PGE 3\Coding for AI\EXO INSIGHTS\DATA\processed\milestone3_cleaning_log.csv
S:\Aivancity\PGE 3\Coding for AI\EXO INSIGHTS\DATA\processed\milestone3_validation_report.csv

Final preprocessed shape: (6451, 15)
Raw source file was not overwritten.


## Milestone 3 — Completion checklist

After running all cells, verify:
- The raw NASA CSV remains unchanged.
- The working dataset contains the selected columns.
- Missing values are documented and not silently imputed.
- Duplicate and physical-value checks are reported.
- No candidate thresholds were applied without approval.
- The cleaned dataset and reports are saved in `DATA/processed/`.

**Next milestone:** Agree on scientifically justified research filters, create analysis-ready subsets, and begin exploratory data analysis.